# World Happiness Report — Data Cleaning
Loads raw CSV, cleans and standardises it, and saves `happiness_clean.csv` for downstream notebooks.

In [12]:
import pandas as pd
import numpy as np

RAW = 'world_happiness_report_2005_2025.csv'
OUT = 'happiness_clean.csv'

df = pd.read_csv(RAW)
print(df.shape)
df.head(3)

(2116, 13)


,year,rank_in_year,country,happiness_score,lower_whisker,upper_whisker,explained_log_gdp_per_capita,explained_social_support,explained_healthy_life_expectancy,explained_freedom,explained_generosity,explained_corruption,dystopia_plus_residual
0,2011,1,Denmark,7.856,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011,2,Finland,7.579,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011,3,Norway,7.524,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1  Schema & types

In [13]:
df['year'] = df['year'].astype(int)
df['rank_in_year'] = df['rank_in_year'].astype(int)

score_cols = [
    'happiness_score', 'lower_whisker', 'upper_whisker',
    'explained_log_gdp_per_capita', 'explained_social_support',
    'explained_healthy_life_expectancy', 'explained_freedom',
    'explained_generosity', 'explained_corruption', 'dystopia_plus_residual'
]
for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.dtypes)

year                                   int64
rank_in_year                           int64
country                               object
happiness_score                      float64
lower_whisker                        float64
upper_whisker                        float64
explained_log_gdp_per_capita         float64
explained_social_support             float64
explained_healthy_life_expectancy    float64
explained_freedom                    float64
explained_generosity                 float64
explained_corruption                 float64
dystopia_plus_residual               float64
dtype: object


## 2  Missing-value audit

In [14]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'missing': missing, 'pct': missing_pct}).query('missing > 0')

,missing,pct
lower_whisker,1094,51.7
upper_whisker,1094,51.7
explained_log_gdp_per_capita,1097,51.8
explained_social_support,1097,51.8
explained_healthy_life_expectancy,1100,52.0
explained_freedom,1099,51.9
explained_generosity,1097,51.8
explained_corruption,1098,51.9
dystopia_plus_residual,1103,52.1


In [15]:
breakdown_cols = [
    'explained_log_gdp_per_capita', 'explained_social_support',
    'explained_healthy_life_expectancy', 'explained_freedom',
    'explained_generosity', 'explained_corruption', 'dystopia_plus_residual'
]
has_breakdown = df[breakdown_cols].notna().all(axis=1)
print('Years WITHOUT breakdown:', sorted(df.loc[~has_breakdown, 'year'].unique()))
print('Years WITH breakdown   :', sorted(df.loc[has_breakdown, 'year'].unique()))

Years WITHOUT breakdown: [np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Years WITH breakdown   : [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 3  Country-name standardisation

In [16]:
name_map = {
    'Republic of Korea': 'South Korea',
    'Viet Nam': 'Vietnam',
    'Lao PDR': 'Laos',
    'Turkiye': 'Turkey',
    'Taiwan Province of China': 'Taiwan',
    'Hong Kong SAR of China': 'Hong Kong',
    'State of Palestine': 'Palestine',
    'Somaliland Region': 'Somaliland',
    'Republic of Moldova': 'Moldova',
    "Cote d'Ivoire": "Cote d'Ivoire",
}
df['country'] = df['country'].replace(name_map)
print('Unique countries:', df['country'].nunique())

Unique countries: 168


## 4  Duplicate check

In [17]:
dupes = df.duplicated(subset=['year', 'country'])
print('Duplicate (year, country) pairs:', dupes.sum())
if dupes.sum():
    display(df[dupes])

Duplicate (year, country) pairs: 0


## 5  Sanity checks

In [18]:
out_of_range = df[(df['happiness_score'] < 0) | (df['happiness_score'] > 10)]
print('Scores outside [0, 10]:', len(out_of_range))

rank_dupes = df.groupby('year')['rank_in_year'].apply(lambda x: x.duplicated().sum())
print('Years with duplicate ranks:', rank_dupes[rank_dupes > 0].to_dict())

sub = df[has_breakdown].copy()
sub['component_sum'] = sub[breakdown_cols].sum(axis=1)
sub['sum_diff'] = (sub['component_sum'] - sub['happiness_score']).abs()
bad_sums = sub[sub['sum_diff'] > 0.05]
print(f'Component-sum deviation >0.05: {len(bad_sums)} rows')

Scores outside [0, 10]: 0
Years with duplicate ranks: {2018: 1}
Component-sum deviation >0.05: 0 rows


## 6  Add helper columns

In [19]:
def tier(rank):
    if rank <= 30:  return 'Top'
    if rank <= 70:  return 'Upper-Mid'
    if rank <= 110: return 'Lower-Mid'
    return 'Bottom'

df['tier'] = df['rank_in_year'].apply(tier)
df['has_breakdown'] = has_breakdown

## 7  Year-over-year change

In [20]:
df = df.sort_values(['country', 'year']).reset_index(drop=True)
df['score_change_yoy'] = df.groupby('country')['happiness_score'].diff()

print('Top 5 biggest gains:')
display(df.nlargest(5, 'score_change_yoy')[['year','country','happiness_score','score_change_yoy']])
print('Top 5 biggest drops:')
display(df.nsmallest(5, 'score_change_yoy')[['year','country','happiness_score','score_change_yoy']])

Top 5 biggest gains:


,year,country,happiness_score,score_change_yoy
43,2012,Angola,5.589,1.411
284,2018,Burundi,3.775,0.870
2103,2012,Zimbabwe,4.827,0.849
996,2023,Kuwait,6.951,0.845
173,2024,Belize,6.711,0.755


Top 5 biggest drops:


,year,country,happiness_score,score_change_yoy
1049,2021,Lebanon,2.955,-1.629
44,2014,Angola,4.033,-1.556
496,2022,DR Congo,3.207,-1.104
1056,2016,Lesotho,3.808,-1.090
1075,2022,Liberia,4.042,-1.080


## 8  Save clean dataset

In [21]:
df.to_csv(OUT, index=False)
print(f'Saved {OUT}  shape: {df.shape}')
df.describe()

Saved happiness_clean.csv  shape: (2116, 16)


,year,rank_in_year,happiness_score,lower_whisker,upper_whisker,explained_log_gdp_per_capita,explained_social_support,explained_healthy_life_expectancy,explained_freedom,explained_generosity,explained_corruption,dystopia_plus_residual,score_change_yoy
count,2116.000000,2116.000000,2116.000000,1022.000000,1022.000000,1019.000000,1019.000000,1016.000000,1017.000000,1019.000000,1018.000000,1013.000000,1948.000000
mean,2018.220227,76.190926,5.465655,5.436091,5.664733,1.265670,1.096746,0.553435,0.609465,0.147343,0.144911,1.736935,0.014664
std,4.249844,43.845101,1.123870,1.140959,1.107424,0.463823,0.357642,0.229980,0.212070,0.084335,0.118803,0.657497,0.221146
min,2011.000000,1.000000,1.364000,1.301000,1.427000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.295000,-1.629000
25%,2015.000000,38.000000,4.604750,4.619707,4.867750,0.944000,0.865000,0.389750,0.471000,0.088000,0.063156,1.305000,-0.071250
50%,2018.000000,76.000000,5.480000,5.592631,5.812000,1.304000,1.140119,0.560500,0.602000,0.134000,0.113000,1.765000,0.016000
75%,2022.000000,114.000000,6.321250,6.290110,6.486500,1.636000,1.382000,0.712325,0.735000,0.195477,0.181330,2.178000,0.124100
max,2025.000000,158.000000,7.856000,7.780000,7.904000,2.209000,1.840000,1.238000,1.147000,0.569814,0.587000,3.482000,1.411000


In [22]:
print('=== DATASET SUMMARY ===')
print(f'Rows          : {len(df)}')
print(f'Years         : {sorted(df["year"].unique())}')
print(f'Countries     : {df["country"].nunique()}')
print(f'Score range   : {df["happiness_score"].min():.3f} to {df["happiness_score"].max():.3f}')
print(f'With breakdown: {df["has_breakdown"].sum()} rows ({df["has_breakdown"].mean()*100:.0f}%)')

=== DATASET SUMMARY ===
Rows          : 2116
Years         : [np.int64(2011), np.int64(2012), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Countries     : 168
Score range   : 1.364 to 7.856
With breakdown: 1013 rows (48%)
